# UCI Bank Marketing — End-to-End ML Walkthrough

> **Business question:** A Portuguese bank ran phone campaigns to sell term deposits (2008–2010).  
> Given 10,000 customers, which 2,000 should we call first to maximize subscriptions?

---

### Notebook structure
1. [Business framing & success metric](#1)
2. [EDA — class imbalance, economic features, leakage trap](#2)
3. [Feature engineering](#3)
4. [Model v1 — *with* `duration` (leaky, benchmark only)](#4)
5. [Model v2 — *without* `duration` (production-safe)](#5)
6. [SHAP — economic interpretation](#6)
7. [Lift curve — business value quantification](#7)


In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys; sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.pipeline import load_data, engineer_features, build_preprocessor, train_model, evaluate, lift_curve_data
from src.eda import plot_eda

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import shap

plt.rcParams['figure.dpi'] = 120
print('Dependencies loaded ✓')

## 1. Business Framing & Success Metric <a id='1'></a>

The bank has **limited call center capacity**. The naive approach — calling everyone — wastes budget on customers who will say no.  

The correct framing is **ranking**, not classification:
- Train a model to output a **probability score** per customer
- Rank all customers by score, descending
- Call the top N

**Primary metric: AUC-ROC** (ranking quality)  
**Secondary metric: F1 on the minority class** (minority-class recall)  
**Business metric: Lift @ top 20%** — how many more subscribers do we get vs random calling?

⚠️ **Accuracy is NOT a useful metric** here. A model that predicts 'no' for everyone achieves ~89% accuracy while being completely useless.

In [ ]:
df = load_data()
print(f'Dataset: {df.shape}')
print(f'Positive rate (subscribed): {(df["y"]=="yes").mean()*100:.1f}%')
df.head(3)

## 2. EDA <a id='2'></a>

In [ ]:
from pathlib import Path
plot_eda(df, Path('../outputs'))
from IPython.display import Image
Image('../outputs/eda_overview.png')

### The `duration` leakage trap

Call duration (`duration`) is the single strongest predictor (correlation ≈ 0.39 with `y`).  
**But it is only known AFTER the call ends** — making it useless for deciding whether to call in the first place.

We train **two models** and document both:
- **v1_with_duration**: AUC will be inflated (~0.93). Shows the trap.
- **v2_no_duration**: Deployable. This is what the bank would actually use.

## 3. Feature Engineering <a id='3'></a>

In [ ]:
df = engineer_features(df)

print('New features added:')
print('  was_contacted_before — flag: was this customer called in a prior campaign?')
print('  season — calendar season derived from month (economic rate cycles are seasonal)')
print('  rate_spread — euribor3m minus cons.price.idx (economic stress proxy)')

X = df.drop(columns=['y'])
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'\nTrain: {len(X_train):,}  |  Test: {len(X_test):,}')

## 4. Model v1 — With Duration (leaky) <a id='4'></a>

In [ ]:
prep_v1 = build_preprocessor(include_duration=True)
X_tr_v1 = prep_v1.fit_transform(X_train)
X_te_v1 = prep_v1.transform(X_test)

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_tr_v1, y_train)

spw = (y_train == 'no').sum() / (y_train == 'yes').sum()
clf_v1 = train_model(X_res, y_res, scale_pos_weight=1.0)

proba_v1, auc_v1, f1_v1 = evaluate(clf_v1, prep_v1, X_test, y_test, 'v1_with_duration')
print('\n⚠️  AUC inflated by duration leakage — this model cannot be deployed')

## 5. Model v2 — Production Model (no duration) <a id='5'></a>

In [ ]:
prep_v2 = build_preprocessor(include_duration=False)
X_tr_v2 = prep_v2.fit_transform(X_train)

sm2 = SMOTE(random_state=42)
X_res2, y_res2 = sm2.fit_resample(X_tr_v2, y_train)

clf_v2 = train_model(X_res2, y_res2, scale_pos_weight=1.0)

proba_v2, auc_v2, f1_v2 = evaluate(clf_v2, prep_v2, X_test, y_test, 'v2_no_duration')

## 6. SHAP — Economic Interpretation <a id='6'></a>

SHAP tells us *why* the model made each prediction, answering two questions:
1. Which features matter globally?
2. Which features drove this specific customer's score?

In [ ]:
X_te_proc = prep_v2.transform(X_test)
feature_names = prep_v2.transformers_[0][2] + prep_v2.transformers_[1][2]

explainer = shap.TreeExplainer(clf_v2)
shap_vals = explainer.shap_values(X_te_proc[:2000])

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
plt.sca(axes[0])
shap.summary_plot(shap_vals, X_te_proc[:2000], feature_names=feature_names,
                  show=False, plot_type='dot', max_display=15)
axes[0].set_title('SHAP Dot — Direction & Magnitude', fontsize=12)

plt.sca(axes[1])
shap.summary_plot(shap_vals, X_te_proc[:2000], feature_names=feature_names,
                  show=False, plot_type='bar', max_display=15)
axes[1].set_title('SHAP Bar — Global Importance', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/shap_analysis.png', dpi=130, bbox_inches='tight')
plt.show()

print('\nKey SHAP insight: high euribor3m DECREASES subscription probability.')
print('When rates are rising, customers keep money liquid → less appetite for term deposits.')

## 7. Lift Curve — Business Value <a id='7'></a>

In [ ]:
pct, lifts, recalls = lift_curve_data(y_test.values, proba_v2)

# Find 20% mark
idx_20 = next(i for i, p in enumerate(pct) if p >= 20)
lift_20 = lifts[idx_20]
recall_20 = recalls[idx_20] * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(pct, lifts, color='#2563EB', lw=2.5, label='Production model')
ax1.axhline(1.0, linestyle='--', color='#9CA3AF', lw=1, label='Random baseline')
ax1.axvline(20, linestyle=':', color='#DC2626', lw=1.5, label='Top 20%')
ax1.scatter([pct[idx_20]], [lift_20], color='#DC2626', s=80, zorder=5)
ax1.annotate(f'  {lift_20:.1f}x lift', (pct[idx_20], lift_20), fontsize=11, color='#DC2626')
ax1.set(xlabel='% Customers Called', ylabel='Lift', title='Cumulative Lift Curve')
ax1.legend()

ax2.plot(pct, [r * 100 for r in recalls], color='#2563EB', lw=2.5, label='Production model')
ax2.plot([0, 100], [0, 100], '--', color='#9CA3AF', lw=1, label='Random')
ax2.axvline(20, linestyle=':', color='#DC2626', lw=1.5)
ax2.scatter([pct[idx_20]], [recall_20], color='#DC2626', s=80, zorder=5)
ax2.annotate(f'  {recall_20:.0f}% of subscribers\ncaptured', 
             (pct[idx_20], recall_20), fontsize=10, color='#DC2626')
ax2.set(xlabel='% Customers Called', ylabel='% Subscribers Captured', title='Recall vs. Effort')
ax2.legend()

plt.tight_layout()
plt.savefig('../outputs/lift_curves.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'\n✅ Calling the top 20% of customers ranked by the model captures')
print(f'   {recall_20:.0f}% of all subscribers — a {lift_20:.1f}x improvement over random calling.')